In [1]:
!pip install PyPDF2 transformers gradio sentencepiece

In [2]:
from google.colab import drive
import os

drive.mount('/content/drive')

pdf_folder_path = '/content/drive/MyDrive/research-paper-analyzer/data/papers/'
uploaded_files = []

if os.path.exists(pdf_folder_path):
    uploaded_files = [
        os.path.join(pdf_folder_path, f)
        for f in os.listdir(pdf_folder_path)
        if f.lower().endswith('.pdf')
    ]
    print(f"✅ {len(uploaded_files)} PDFs found!")
    for i, f in enumerate(uploaded_files):
        print(f"  {i+1}. {os.path.basename(f)}")
else:
    print("❌ Folder not found. Check your path.")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
✅ 18 PDFs found!
  1. A HORMONE-INSPIRED EMOTION LAYER.pdf
  2. Not All Symbols Are Equal.pdf
  3. Embedded Made Easy.pdf
  4. Earth Science Foundation Models.pdf
  5. Clinical Timeline Reconstruction RAG.pdf
  6. Discovery-Oriented Faceting.pdf
  7. Explainable Detection of Depression Status Shifts.pdf
  8. Evo-Depth.pdf
  9. Exploring Vision-Language Models.pdf
  10. LLMGenerated Political Discourse.pdf
  11. LATERN.pdf
  12. HDRFace.pdf
  13. MemEye.pdf
  14. nASR An End To End Trainable Layer.pdf
  15. Multimodal Abstractive Summarization.pdf
  16. MILM.pdf
  17. MEMLENS.pdf
  18. Towards Visually Grounded Multimodal Summarization.pdf


In [3]:
import PyPDF2
import os
import re

In [4]:

def extract_text_from_pdf(pdf_path):
    text = ""
    try:
        with open(pdf_path, "rb") as file:
            reader = PyPDF2.PdfReader(file)
            for page in reader.pages:
                page_text = page.extract_text()
                if page_text:
                    text += page_text + "\n"
    except Exception as e:
        print(f"  ❌ Error: {e}")
    return text

# ── Clean ─────────────────────────────────────────────────
def clean_text(text):
    text = text.encode("ascii", "ignore").decode()
    text = re.sub(r'http\S+|www\S+', '', text)
    text = re.sub(r'\S+@\S+', '', text)
    text = re.sub(r'[ \t]+', ' ', text)
    text = re.sub(r'^\s*\d+\s*$', '', text, flags=re.MULTILINE)
    lines = text.split('\n')
    lines = [line.strip() for line in lines if len(line.strip()) > 4]
    text = '\n'.join(lines)
    text = re.sub(r'\n{3,}', '\n\n', text)
    return text.strip()

# ── Section Splitter ─────────────────────────────────────
def split_into_sections(text):
    sections = {
        "abstract": "",
        "introduction": "",
        "methodology": "",
        "results": "",
        "conclusion": "",
        "full_text": text
    }
    patterns = {
        "abstract":     r'(abstract)(.*?)(introduction|1\.)',
        "introduction": r'(introduction|1\.?\s+introduction)(.*?)(2\.|method|related)',
        "methodology":  r'(method|methodology|approach|proposed)(.*?)(result|experiment|evaluation)',
        "results":      r'(result|experiment|evaluation)(.*?)(conclusion|discussion|limitation)',
        "conclusion":   r'(conclusion|discussion|limitation)(.*?)($|reference|bibliograph)',
    }
    text_lower = text.lower()
    for section_name, pattern in patterns.items():
        match = re.search(pattern, text_lower, re.DOTALL | re.IGNORECASE)
        if match:
            start = match.start(2)
            end = match.end(2)
            sections[section_name] = text[start:end].strip()
    return sections

# ── Process ALL papers ────────────────────────────────────
all_papers = {}
all_papers_cleaned = {}
all_papers_sections = {}

print("Processing all papers...\n")
for pdf_path in uploaded_files:
    filename = os.path.basename(pdf_path)
    raw = extract_text_from_pdf(pdf_path)
    cleaned = clean_text(raw)
    sections = split_into_sections(cleaned)
    all_papers[filename] = raw
    all_papers_cleaned[filename] = cleaned
    all_papers_sections[filename] = sections
    print(f"✅ {filename[:55]}")

print(f"\n✅ All {len(all_papers_sections)} papers ready!")

Processing all papers...

✅ A HORMONE-INSPIRED EMOTION LAYER.pdf
✅ Not All Symbols Are Equal.pdf
✅ Embedded Made Easy.pdf
✅ Earth Science Foundation Models.pdf
✅ Clinical Timeline Reconstruction RAG.pdf
✅ Discovery-Oriented Faceting.pdf
✅ Explainable Detection of Depression Status Shifts.pdf
✅ Evo-Depth.pdf
✅ Exploring Vision-Language Models.pdf
✅ LLMGenerated Political Discourse.pdf
✅ LATERN.pdf
✅ HDRFace.pdf
✅ MemEye.pdf
✅ nASR An End To End Trainable Layer.pdf
✅ Multimodal Abstractive Summarization.pdf
✅ MILM.pdf
✅ MEMLENS.pdf
✅ Towards Visually Grounded Multimodal Summarization.pdf

✅ All 18 papers ready!


 Load DistilBART Model

In [5]:
from transformers import pipeline
import warnings
warnings.filterwarnings("ignore")

print("Loading DistilBART model... please wait")

summarizer = pipeline(
    "summarization",
    model="sshleifer/distilbart-cnn-12-6",
    device=-1
)

print("✅ Model loaded successfully!")

The cache for model files in Transformers v4.22.0 has been updated. Migrating your old cache. This is a one-time only operation. You can interrupt this and resume the migration later on by calling `transformers.utils.move_cache()`.


0it [00:00, ?it/s]

Loading DistilBART model... please wait


pytorch_model.bin:   0%|          | 0.00/1.22G [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

✅ Model loaded successfully!


Summarization Function

In [9]:
def summarize_text(text, max_length=150, min_length=50):
    if not text or len(text.strip()) < 100:
        return "Section not found or too short."

    # Trim to 900 words to stay within token limit
    words = text.split()
    if len(words) > 900:
        text = " ".join(words[:900])

    try:
        result = summarizer(
            text,
            max_length=max_length,
            min_length=min_length,
            do_sample=False,
            truncation=True
        )
        return result[0]["summary_text"]

    except Exception as e:
        return f"Could not summarize: {str(e)}"

print("✅ Summarization function ready!")

✅ Summarization function ready!


Test on One Paper

In [10]:
test_paper = list(all_papers_sections.keys())[0]
sections = all_papers_sections[test_paper]

print(f"📄 Paper: {test_paper}")
print("=" * 60)

print("\n📌 ABSTRACT SUMMARY:")
print(summarize_text(sections["abstract"]))

print("\n🔍 INTRODUCTION SUMMARY:")
print(summarize_text(sections["introduction"]))

print("\n⚙️  METHODOLOGY SUMMARY:")
print(summarize_text(sections["methodology"]))

print("\n📊 RESULTS SUMMARY:")
print(summarize_text(sections["results"]))

print("\n✅ CONCLUSION SUMMARY:")
print(summarize_text(sections["conclusion"]))

📄 Paper: A HORMONE-INSPIRED EMOTION LAYER.pdf

📌 ABSTRACT SUMMARY:
 Large Language Models have demonstrated remarkable capabilities in generating contextually relevant and grammatically correct text . They fundamentally lack the ability to process and respond to emotional context in a manner analogous to human emotional cognition . We introduce HormoneT5, a novel architecture that combines transformer language models with a biologically-inspiredHormone Emotion Block that simulates the human endocrine systems role in emotional processing .

🔍 INTRODUCTION SUMMARY:
 Modern Large Language Models (LLMs) have achieved unprecedented success in natural language understanding and generation . Foundational models such as GPT, T5 and BERT demonstrate remarkable proficiency in tasks ranging from translation and summarization to question answering and open-ended dialogue . Despite these advanced linguistic capabilities, LLMs remain fundamentally constrained . Processing is rooted in statistical pa

Full Analyze Function (Test on 3 Papers)

In [11]:
def analyze_paper(paper_name):
    if paper_name not in all_papers_sections:
        return "Paper not found."

    sections = all_papers_sections[paper_name]

    print(f"\n{'='*60}")
    print(f"📄 {paper_name}")
    print("=" * 60)

    labels = {
        "abstract":     "📌 What is this paper about?",
        "introduction": "🔍 What problem does it solve?",
        "methodology":  "⚙️  How did they solve it?",
        "results":      "📊 What did they find?",
        "conclusion":   "✅ Final Takeaway"
    }

    analysis = {}
    for section, label in labels.items():
        print(f"\n{label}")
        print("-" * 40)
        summary = summarize_text(sections[section])
        print(summary)
        analysis[section] = summary

    return analysis

# Test on first 3 papers
for paper in list(all_papers_sections.keys())[:3]:
    analyze_paper(paper)


📄 A HORMONE-INSPIRED EMOTION LAYER.pdf

📌 What is this paper about?
----------------------------------------
 Large Language Models have demonstrated remarkable capabilities in generating contextually relevant and grammatically correct text . They fundamentally lack the ability to process and respond to emotional context in a manner analogous to human emotional cognition . We introduce HormoneT5, a novel architecture that combines transformer language models with a biologically-inspiredHormone Emotion Block that simulates the human endocrine systems role in emotional processing .

🔍 What problem does it solve?
----------------------------------------
 Modern Large Language Models (LLMs) have achieved unprecedented success in natural language understanding and generation . Foundational models such as GPT, T5 and BERT demonstrate remarkable proficiency in tasks ranging from translation and summarization to question answering and open-ended dialogue . Despite these advanced linguistic ca

Gradio Interface.

In [12]:
import gradio as gr
import tempfile
import os

def analyze_uploaded_paper(pdf_file):
    """
    Takes an uploaded PDF file and returns full analysis.
    This is what recruiters will see and interact with.
    """
    if pdf_file is None:
        return "⚠️ Please upload a PDF file.", "", "", "", ""

    try:
        # Extract text from uploaded file
        raw_text = extract_text_from_pdf(pdf_file.name)

        if not raw_text or len(raw_text.strip()) < 200:
            return "❌ Could not extract text from this PDF.", "", "", "", ""

        # Clean the text
        cleaned = clean_text(raw_text)

        # Split into sections
        sections = split_into_sections(cleaned)

        # Summarize each section
        abstract_summary    = summarize_text(sections["abstract"])
        intro_summary       = summarize_text(sections["introduction"])
        methodology_summary = summarize_text(sections["methodology"])
        results_summary     = summarize_text(sections["results"])
        conclusion_summary  = summarize_text(sections["conclusion"])

        return (
            abstract_summary,
            intro_summary,
            methodology_summary,
            results_summary,
            conclusion_summary
        )

    except Exception as e:
        return f"❌ Error: {str(e)}", "", "", "", ""


# ── Build the Interface ───────────────────────────────────
with gr.Blocks(title="Research Paper Analyzer") as demo:

    gr.Markdown("# 📄 Research Paper Analyzer")
    gr.Markdown("Upload any research paper PDF and get an instant AI-powered summary of all key sections.")

    with gr.Row():
        pdf_input = gr.File(
            label="📂 Upload Research Paper (PDF)",
            file_types=[".pdf"]
        )

    analyze_btn = gr.Button("🔍 Analyze Paper", variant="primary")

    gr.Markdown("---")
    gr.Markdown("## 📊 Analysis Results")

    with gr.Row():
        with gr.Column():
            abstract_out = gr.Textbox(
                label="📌 What is this paper about?",
                lines=4,
                interactive=False
            )
            intro_out = gr.Textbox(
                label="🔍 What problem does it solve?",
                lines=4,
                interactive=False
            )
            methodology_out = gr.Textbox(
                label="⚙️ How did they solve it?",
                lines=4,
                interactive=False
            )

        with gr.Column():
            results_out = gr.Textbox(
                label="📈 What did they find?",
                lines=4,
                interactive=False
            )
            conclusion_out = gr.Textbox(
                label="✅ Final Takeaway",
                lines=4,
                interactive=False
            )

    analyze_btn.click(
        fn=analyze_uploaded_paper,
        inputs=[pdf_input],
        outputs=[
            abstract_out,
            intro_out,
            methodology_out,
            results_out,
            conclusion_out
        ]
    )

    gr.Markdown("---")
    gr.Markdown("*Built by Warda | MS Artificial Intelligence | NLP & Transformer Models*")

# Launch with public link
demo.launch(share=True)

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://1e4736adf86a18c157.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
